<a href="https://colab.research.google.com/github/rachitt-t/capstone_project/blob/main/Copy_of_project_capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

STEP 1 (DATASET UPLOAD)

In [3]:
import pandas as pd

file_path = "movie_lines.txt"

lines = []

with open(file_path, 'r', encoding='iso-8859-1') as f:
    for line in f:
        parts = line.split(" +++$+++ ")

        if len(parts) == 5:
            character = parts[3].strip()
            dialogue = parts[4].strip()

            if len(dialogue.split()) > 3:
                lines.append([character, dialogue])

df = pd.DataFrame(lines, columns=["Character", "Dialogue"])

# Shuffle and select 2800 rows
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df = df.iloc[:2800]

df.to_csv("movie_dialogues_2800.csv", index=False)

print(df.head())
print("Total rows:", len(df))

       Character                                           Dialogue
0        SALIERI           Dear Mozart, my sincere congratulations.
1        MARYLIN        Thank you for the coffee. It's very robust.
2          KAREN       I married Martin. That was a full- time job.
3  MRS. ROBINSON  Because I don't feel safe until I get the ligh...
4         NATHAN             Nothing. Hard day. Gonna have a drink.
Total rows: 2800


STEP 2(Sentiment Labeling)

In [4]:
!pip install transformers

In [5]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [6]:
import pandas as pd

df = pd.read_csv("movie_dialogues_2800.csv")
df.head()

,Character,Dialogue
0,SALIERI,"Dear Mozart, my sincere congratulations."
1,MARYLIN,Thank you for the coffee. It's very robust.
2,KAREN,I married Martin. That was a full- time job.
3,MRS. ROBINSON,Because I don't feel safe until I get the ligh...
4,NATHAN,Nothing. Hard day. Gonna have a drink.


In [7]:
def get_sentiment(text):
    result = classifier(text[:512])[0]   # limit for safety

    if result['label'] == 'POSITIVE':
        return "Positive"
    elif result['label'] == 'NEGATIVE':
        return "Negative"
    else:
        return "Neutral"

df['Sentiment'] = df['Dialogue'].apply(get_sentiment)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [8]:
df.to_csv("movie_dialogues_with_sentiment.csv", index=False)

In [9]:
from google.colab import files
files.download("movie_dialogues_with_sentiment.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

step 3(Train-Test Split + Data Preparation)

In [10]:
import pandas as pd

df = pd.read_csv("movie_dialogues_with_sentiment.csv")
df.head()

,Character,Dialogue,Sentiment
0,SALIERI,"Dear Mozart, my sincere congratulations.",Positive
1,MARYLIN,Thank you for the coffee. It's very robust.,Positive
2,KAREN,I married Martin. That was a full- time job.,Positive
3,MRS. ROBINSON,Because I don't feel safe until I get the ligh...,Negative
4,NATHAN,Nothing. Hard day. Gonna have a drink.,Negative


In [11]:
label_map = {
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2
}

df['Label'] = df['Sentiment'].map(label_map)

In [12]:
X = df['Dialogue']
y = df['Label']

In [13]:
from sklearn.model_selection import train_test_split

X_train_80, X_test_80, y_train_80, y_test_80 = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [14]:
X_train_90, X_test_90, y_train_90, y_test_90 = train_test_split(
    X, y, test_size=0.1, random_state=42
)

In [15]:
print("80:20 Split")
print("Train:", len(X_train_80), "Test:", len(X_test_80))

print("\n90:10 Split")
print("Train:", len(X_train_90), "Test:", len(X_test_90))

80:20 Split
Train: 2240 Test: 560

90:10 Split
Train: 2520 Test: 280


step 4(Text Preprocessing)

In [16]:
!pip install transformers torch scikit-learn

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [17]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [18]:
X_train_tokens = tokenizer(
    list(X_train_80),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

X_test_tokens = tokenizer(
    list(X_test_80),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

In [19]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [20]:
train_dataset = Dataset(X_train_tokens, list(y_train_80))
test_dataset = Dataset(X_test_tokens, list(y_test_80))

step 5(training transformer model)

In [21]:
import torch
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

In [22]:
df = pd.read_csv("movie_dialogues_with_sentiment.csv")
df.head()

,Character,Dialogue,Sentiment
0,SALIERI,"Dear Mozart, my sincere congratulations.",Positive
1,MARYLIN,Thank you for the coffee. It's very robust.,Positive
2,KAREN,I married Martin. That was a full- time job.,Positive
3,MRS. ROBINSON,Because I don't feel safe until I get the ligh...,Negative
4,NATHAN,Nothing. Hard day. Gonna have a drink.,Negative


BERT MODEL

In [23]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
def compute_metrics(pred):
    logits, labels = pred
    preds = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

BERT (3 EPOCHS)

In [25]:
training_args = TrainingArguments(
    output_dir="./bert_results_3",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./bert_logs_3",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [26]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [27]:
trainer.train()

Step,Training Loss
500,0.403644


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=840, training_loss=0.2656226612272717, metrics={'train_runtime': 183.0244, 'train_samples_per_second': 36.716, 'train_steps_per_second': 4.59, 'total_flos': 442030541783040.0, 'train_loss': 0.2656226612272717, 'epoch': 3.0})

In [28]:
results_bert_3 = trainer.evaluate()
print("BERT (3 epochs):", results_bert_3)

BERT (3 epochs): {'eval_loss': 0.970231294631958, 'eval_accuracy': 0.8214285714285714, 'eval_f1': 0.8209378868858893, 'eval_precision': 0.8207908163265305, 'eval_recall': 0.8214285714285714, 'eval_runtime': 4.0548, 'eval_samples_per_second': 138.106, 'eval_steps_per_second': 17.263, 'epoch': 3.0}


BERT (5 EPOCHS)

In [29]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [30]:
training_args = TrainingArguments(
    output_dir="./bert_results_5",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./bert_logs_5",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [31]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

results_bert_5 = trainer.evaluate()
print("BERT (5 epochs):", results_bert_5)

Step,Training Loss
500,0.420029
1000,0.073194


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BERT (5 epochs): {'eval_loss': 1.2225956916809082, 'eval_accuracy': 0.8339285714285715, 'eval_f1': 0.8327205279115392, 'eval_precision': 0.8335103802449664, 'eval_recall': 0.8339285714285715, 'eval_runtime': 4.0041, 'eval_samples_per_second': 139.857, 'eval_steps_per_second': 17.482, 'epoch': 5.0}


BERT (7 EPOCHS)

In [32]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [33]:
training_args = TrainingArguments(
    output_dir="./bert_results_7",
    num_train_epochs=7,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./bert_logs_7",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [34]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

results_bert_7 = trainer.evaluate()
print("BERT (7 epochs):", results_bert_7)

Step,Training Loss
500,0.408642
1000,0.089209
1500,0.022092


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BERT (7 epochs): {'eval_loss': 1.2058250904083252, 'eval_accuracy': 0.8357142857142857, 'eval_f1': 0.8345928292941639, 'eval_precision': 0.8352662895421256, 'eval_recall': 0.8357142857142857, 'eval_runtime': 4.0573, 'eval_samples_per_second': 138.022, 'eval_steps_per_second': 17.253, 'epoch': 7.0}


RoBERTa MODEL

In [35]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3
)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [36]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [37]:
X_train_tokens = tokenizer(
    list(X_train_80),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

X_test_tokens = tokenizer(
    list(X_test_80),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

In [38]:
train_dataset = Dataset(X_train_tokens, list(y_train_80))
test_dataset = Dataset(X_test_tokens, list(y_test_80))

RoBERTa (3 epochs)

In [39]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./roberta_results_3",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./roberta_logs_3",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [40]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)
trainer.train()
results_roberta_3 = trainer.evaluate()
print("RoBERTa (3 epochs):", results_roberta_3)

Step,Training Loss
500,0.512854


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RoBERTa (3 epochs): {'eval_loss': 0.9353940486907959, 'eval_accuracy': 0.8089285714285714, 'eval_f1': 0.8081842711567585, 'eval_precision': 0.8080809178674874, 'eval_recall': 0.8089285714285714, 'eval_runtime': 3.7425, 'eval_samples_per_second': 149.632, 'eval_steps_per_second': 18.704, 'epoch': 3.0}


RoBERTa (5 epochs)

In [41]:
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [42]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./roberta_results_5",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./roberta_logs_5",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [43]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)
trainer.train()
results_roberta_5 = trainer.evaluate()
print("RoBERTa (5 epochs):", results_roberta_5)

Step,Training Loss
500,0.542636
1000,0.276113


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RoBERTa (5 epochs): {'eval_loss': 0.9401739835739136, 'eval_accuracy': 0.8339285714285715, 'eval_f1': 0.8337662166667529, 'eval_precision': 0.8336527244045814, 'eval_recall': 0.8339285714285715, 'eval_runtime': 3.7782, 'eval_samples_per_second': 148.217, 'eval_steps_per_second': 18.527, 'epoch': 5.0}


RoBERTa (7 epochs)

In [44]:
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [45]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./roberta_results_7",
    num_train_epochs=7,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./roberta_logs_7",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [46]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)
trainer.train()
results_roberta_7 = trainer.evaluate()
print("RoBERTa (7 epochs):", results_roberta_7)

Step,Training Loss
500,0.551101
1000,0.300711
1500,0.084020


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RoBERTa (7 epochs): {'eval_loss': 1.3303643465042114, 'eval_accuracy': 0.8196428571428571, 'eval_f1': 0.8201043124949526, 'eval_precision': 0.8209931004751437, 'eval_recall': 0.8196428571428571, 'eval_runtime': 3.8083, 'eval_samples_per_second': 147.048, 'eval_steps_per_second': 18.381, 'epoch': 7.0}


DeBERTa MODEL

In [47]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-base",
    num_labels=3
)

config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/559M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/559M [00:00<?, ?B/s]

DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [48]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

In [49]:
X_train_tokens = tokenizer(
    list(X_train_80),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

X_test_tokens = tokenizer(
    list(X_test_80),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

In [50]:
train_dataset = Dataset(X_train_tokens, list(y_train_80))
test_dataset = Dataset(X_test_tokens, list(y_test_80))

DeBERTa (3 epochs)

In [51]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./deberta_results_3",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./deberta_logs_3",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [52]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)
trainer.train()
results_deberta_3 = trainer.evaluate()
print("DeBERTa (3 epochs):", results_deberta_3)

Step,Training Loss
500,0.499004


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DeBERTa (3 epochs): {'eval_loss': 1.0061529874801636, 'eval_accuracy': 0.7875, 'eval_f1': 0.7889640795278157, 'eval_precision': 0.7965653002006876, 'eval_recall': 0.7875, 'eval_runtime': 5.1341, 'eval_samples_per_second': 109.075, 'eval_steps_per_second': 13.634, 'epoch': 3.0}


DeBERTa (5 epochs)

In [53]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-base",
    num_labels=3
)

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [54]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./deberta_results_5",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./deberta_logs_5",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [55]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

results_deberta_5 = trainer.evaluate()
print("DeBERTa (5 epochs):", results_deberta_5)

Step,Training Loss
500,0.537023
1000,0.224431


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DeBERTa (5 epochs): {'eval_loss': 1.2419463396072388, 'eval_accuracy': 0.7875, 'eval_f1': 0.7868363215731637, 'eval_precision': 0.7865883354845576, 'eval_recall': 0.7875, 'eval_runtime': 5.1772, 'eval_samples_per_second': 108.166, 'eval_steps_per_second': 13.521, 'epoch': 5.0}


DeBERTa (7 epochs)

In [56]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-base",
    num_labels=3
)

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [57]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./deberta_results_7",
    num_train_epochs=7,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./deberta_logs_7",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [58]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

results_deberta_7 = trainer.evaluate()
print("DeBERTa (7 epochs):", results_deberta_7)

Step,Training Loss
500,0.548659
1000,0.279949
1500,0.101182


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DeBERTa (7 epochs): {'eval_loss': 1.3205790519714355, 'eval_accuracy': 0.8125, 'eval_f1': 0.8111360799001248, 'eval_precision': 0.8117127355873955, 'eval_recall': 0.8125, 'eval_runtime': 5.1436, 'eval_samples_per_second': 108.873, 'eval_steps_per_second': 13.609, 'epoch': 7.0}


DistilBERT MODEL

In [59]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [60]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [61]:
X_train_tokens = tokenizer(list(X_train_80), padding=True, truncation=True, max_length=128, return_tensors="pt")
X_test_tokens = tokenizer(list(X_test_80), padding=True, truncation=True, max_length=128, return_tensors="pt")

In [62]:
train_dataset = Dataset(X_train_tokens, list(y_train_80))
test_dataset = Dataset(X_test_tokens, list(y_test_80))

DistilBERT (3 Epochs)

In [63]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distilbert_results_3",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./distilbert_logs_3",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [64]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

results_distilbert_3 = trainer.evaluate()
print("DistilBERT (3 epochs):", results_distilbert_3)

Step,Training Loss
500,0.388084


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DistilBERT (3 epochs): {'eval_loss': 0.7476029992103577, 'eval_accuracy': 0.85, 'eval_f1': 0.8504212877689383, 'eval_precision': 0.8513904049067, 'eval_recall': 0.85, 'eval_runtime': 2.0952, 'eval_samples_per_second': 267.28, 'eval_steps_per_second': 33.41, 'epoch': 3.0}


DistilBERT (5 Epochs)

In [65]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [66]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distilbert_results_5",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./distilbert_logs_5",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [67]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

results_distilbert_5 = trainer.evaluate()
print("DistilBERT (5 epochs):", results_distilbert_5)

Step,Training Loss
500,0.407205
1000,0.076056


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DistilBERT (5 epochs): {'eval_loss': 1.0847197771072388, 'eval_accuracy': 0.8357142857142857, 'eval_f1': 0.8352628559350184, 'eval_precision': 0.8351615646258503, 'eval_recall': 0.8357142857142857, 'eval_runtime': 2.0995, 'eval_samples_per_second': 266.73, 'eval_steps_per_second': 33.341, 'epoch': 5.0}


DistilBERT (7 Epochs)

In [68]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [69]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distilbert_results_7",
    num_train_epochs=7,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./distilbert_logs_7",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [70]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

results_distilbert_7 = trainer.evaluate()
print("DistilBERT (7 epochs):", results_distilbert_7)

Step,Training Loss
500,0.393480
1000,0.094827
1500,0.018254


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DistilBERT (7 epochs): {'eval_loss': 1.2071970701217651, 'eval_accuracy': 0.8267857142857142, 'eval_f1': 0.8264974034663453, 'eval_precision': 0.8263438391191694, 'eval_recall': 0.8267857142857142, 'eval_runtime': 2.082, 'eval_samples_per_second': 268.971, 'eval_steps_per_second': 33.621, 'epoch': 7.0}


ELECTRA

In [71]:
model = AutoModelForSequenceClassification.from_pretrained(
    "google/electra-base-discriminator",
    num_labels=3
)

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

ElectraForSequenceClassification LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
electra.embeddings_project.bias                   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
electra.embeddings_project.weight                 | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- 

In [72]:
tokenizer = AutoTokenizer.from_pretrained("google/electra-base-discriminator")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [73]:
X_train_tokens = tokenizer(list(X_train_80), padding=True, truncation=True, max_length=128, return_tensors="pt")
X_test_tokens = tokenizer(list(X_test_80), padding=True, truncation=True, max_length=128, return_tensors="pt")

In [74]:
train_dataset = Dataset(X_train_tokens, list(y_train_80))
test_dataset = Dataset(X_test_tokens, list(y_test_80))

ELECTRA (3 Epochs)

In [75]:
training_args = TrainingArguments(
    output_dir="./electra_results_3",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./electra_logs_3",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [76]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

results_electra_3 = trainer.evaluate()
print("ELECTRA (3 epochs):", results_electra_3)

Step,Training Loss
500,0.479449


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

ELECTRA (3 epochs): {'eval_loss': 0.6865459680557251, 'eval_accuracy': 0.8357142857142857, 'eval_f1': 0.8338016294559027, 'eval_precision': 0.8362981147515185, 'eval_recall': 0.8357142857142857, 'eval_runtime': 3.7728, 'eval_samples_per_second': 148.43, 'eval_steps_per_second': 18.554, 'epoch': 3.0}


ELECTRA (5 Epochs)

In [77]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "google/electra-base-discriminator",
    num_labels=3
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
electra.embeddings_project.bias                   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
electra.embeddings_project.weight                 | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- 

In [78]:
training_args = TrainingArguments(
    output_dir="./electra_results_5",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./electra_logs_5",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [79]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

results_electra_5 = trainer.evaluate()
print("ELECTRA (5 epochs):", results_electra_5)

Step,Training Loss
500,0.481586
1000,0.176595


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

ELECTRA (5 epochs): {'eval_loss': 0.9956471920013428, 'eval_accuracy': 0.8267857142857142, 'eval_f1': 0.8271387398452065, 'eval_precision': 0.8277530431090826, 'eval_recall': 0.8267857142857142, 'eval_runtime': 3.7604, 'eval_samples_per_second': 148.919, 'eval_steps_per_second': 18.615, 'epoch': 5.0}


ELECTRA (7 Epochs)

In [80]:
model = AutoModelForSequenceClassification.from_pretrained(
    "google/electra-base-discriminator",
    num_labels=3
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
electra.embeddings_project.bias                   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
electra.embeddings_project.weight                 | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- 

In [81]:
training_args = TrainingArguments(
    output_dir="./electra_results_7",
    num_train_epochs=7,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./electra_logs_7",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [82]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

results_electra_7 = trainer.evaluate()
print("ELECTRA (7 epochs):", results_electra_7)

Step,Training Loss
500,0.691472
1000,0.677016
1500,0.677700


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

ELECTRA (7 epochs): {'eval_loss': 0.6713536977767944, 'eval_accuracy': 0.5857142857142857, 'eval_f1': 0.4326898326898327, 'eval_precision': 0.3430612244897959, 'eval_recall': 0.5857142857142857, 'eval_runtime': 3.7383, 'eval_samples_per_second': 149.8, 'eval_steps_per_second': 18.725, 'epoch': 7.0}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


SVM KERNELS

In [83]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf_80 = vectorizer.fit_transform(X_train_80)
X_test_tfidf_80 = vectorizer.transform(X_test_80)

In [84]:
from sklearn.svm import SVC

LINEAR KERNEL

In [85]:
svm_linear = SVC(kernel='linear')
svm_linear.fit(X_train_tfidf_80, y_train_80)
y_pred_linear = svm_linear.predict(X_test_tfidf_80)

POLYNOMIAL KERNEL

In [86]:
svm_poly = SVC(kernel='poly')
svm_poly.fit(X_train_tfidf_80, y_train_80)
y_pred_poly = svm_poly.predict(X_test_tfidf_80)

RBF (Gaussian) Kernel

In [87]:
svm_rbf = SVC(kernel='rbf')

svm_rbf.fit(X_train_tfidf_80, y_train_80)

y_pred_rbf = svm_rbf.predict(X_test_tfidf_80)

EVALUATION

In [88]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

LINEAR RESULTS

In [89]:
print("Linear SVM")
print("Accuracy:", accuracy_score(y_test_80, y_pred_linear))
print("Precision:", precision_score(y_test_80, y_pred_linear, average='weighted'))
print("Recall:", recall_score(y_test_80, y_pred_linear, average='weighted'))
print("F1 Score:", f1_score(y_test_80, y_pred_linear, average='weighted'))

Linear SVM
Accuracy: 0.6857142857142857
Precision: 0.6818317099567099
Recall: 0.6857142857142857
F1 Score: 0.676505208825103


POLYNOMIAL RESULTS

In [90]:
print("Polynomial SVM")
print("Accuracy:", accuracy_score(y_test_80, y_pred_poly))
print("Precision:", precision_score(y_test_80, y_pred_poly, average='weighted'))
print("Recall:", recall_score(y_test_80, y_pred_poly, average='weighted'))
print("F1 Score:", f1_score(y_test_80, y_pred_poly, average='weighted'))

Polynomial SVM
Accuracy: 0.6142857142857143
Precision: 0.6723629829290206
Recall: 0.6142857142857143
F1 Score: 0.5109989247393827


RBF RESULTS

In [91]:
print("RBF SVM")
print("Accuracy:", accuracy_score(y_test_80, y_pred_rbf))
print("Precision:", precision_score(y_test_80, y_pred_rbf, average='weighted'))
print("Recall:", recall_score(y_test_80, y_pred_rbf, average='weighted'))
print("F1 Score:", f1_score(y_test_80, y_pred_rbf, average='weighted'))

RBF SVM
Accuracy: 0.6642857142857143
Precision: 0.6654587273191924
Recall: 0.6642857142857143
F1 Score: 0.6395765869140116


VOTING CLASSIFIER

In [92]:
from sklearn.ensemble import VotingClassifier

voting_clf = VotingClassifier(
    estimators=[
        ('linear', svm_linear),
        ('poly', svm_poly),
        ('rbf', svm_rbf)
    ],
    voting='hard'
)

voting_clf.fit(X_train_tfidf_80, y_train_80)

y_pred_vote = voting_clf.predict(X_test_tfidf_80)

In [93]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Voting Classifier")
print("Accuracy:", accuracy_score(y_test_80, y_pred_vote))
print("Precision:", precision_score(y_test_80, y_pred_vote, average='weighted'))
print("Recall:", recall_score(y_test_80, y_pred_vote, average='weighted'))
print("F1 Score:", f1_score(y_test_80, y_pred_vote, average='weighted'))

Voting Classifier
Accuracy: 0.6607142857142857
Precision: 0.6615203373015873
Recall: 0.6607142857142857
F1 Score: 0.6349206349206349


CONFUSION MATRIX

In [94]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_80, y_pred_vote)
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[285  43]
 [147  85]]


MCC (Matthews Correlation Coefficient)

In [95]:
from sklearn.metrics import matthews_corrcoef

mcc = matthews_corrcoef(y_test_80, y_pred_vote)
print("MCC:", mcc)

MCC: 0.2760083981756184


RMSE (Root Mean Square Error)

In [96]:
from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test_80, y_pred_vote))
print("RMSE:", rmse)

RMSE: 1.164964745021435


RESULTS

In [97]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

data = {
    "Model": [
        "BERT (3)", "BERT (5)", "BERT (7)",
        "RoBERTa (3)", "RoBERTa (5)", "RoBERTa (7)",
        "DeBERTa (3)", "DeBERTa (5)", "DeBERTa (7)",
        "DistilBERT (3)",
        "ELECTRA (3)", "ELECTRA (5)", "ELECTRA (7)",
        "SVM Linear", "SVM Poly", "SVM RBF", "Voting"
    ],

    "Accuracy": [
        results_bert_3['eval_accuracy'],
        results_bert_5['eval_accuracy'],
        results_bert_7['eval_accuracy'],

        results_roberta_3['eval_accuracy'],
        results_roberta_5['eval_accuracy'],
        results_roberta_7['eval_accuracy'],

        results_deberta_3['eval_accuracy'],
        results_deberta_5['eval_accuracy'],
        results_deberta_7['eval_accuracy'],

        results_distilbert_3['eval_accuracy'],

        results_electra_3['eval_accuracy'],
        results_electra_5['eval_accuracy'],
        results_electra_7['eval_accuracy'],

        accuracy_score(y_test_80, y_pred_linear),
        accuracy_score(y_test_80, y_pred_poly),
        accuracy_score(y_test_80, y_pred_rbf),
        accuracy_score(y_test_80, y_pred_vote)
    ],

    "Precision": [
        results_bert_3['eval_precision'],
        results_bert_5['eval_precision'],
        results_bert_7['eval_precision'],

        results_roberta_3['eval_precision'],
        results_roberta_5['eval_precision'],
        results_roberta_7['eval_precision'],

        results_deberta_3['eval_precision'],
        results_deberta_5['eval_precision'],
        results_deberta_7['eval_precision'],

        results_distilbert_3['eval_precision'],

        results_electra_3['eval_precision'],
        results_electra_5['eval_precision'],
        results_electra_7['eval_precision'],

        precision_score(y_test_80, y_pred_linear, average='weighted'),
        precision_score(y_test_80, y_pred_poly, average='weighted'),
        precision_score(y_test_80, y_pred_rbf, average='weighted'),
        precision_score(y_test_80, y_pred_vote, average='weighted')
    ],

    "Recall": [
        results_bert_3['eval_recall'],
        results_bert_5['eval_recall'],
        results_bert_7['eval_recall'],

        results_roberta_3['eval_recall'],
        results_roberta_5['eval_recall'],
        results_roberta_7['eval_recall'],

        results_deberta_3['eval_recall'],
        results_deberta_5['eval_recall'],
        results_deberta_7['eval_recall'],

        results_distilbert_3['eval_recall'],

        results_electra_3['eval_recall'],
        results_electra_5['eval_recall'],
        results_electra_7['eval_recall'],

        recall_score(y_test_80, y_pred_linear, average='weighted'),
        recall_score(y_test_80, y_pred_poly, average='weighted'),
        recall_score(y_test_80, y_pred_rbf, average='weighted'),
        recall_score(y_test_80, y_pred_vote, average='weighted')
    ],

    "F1 Score": [
        results_bert_3['eval_f1'],
        results_bert_5['eval_f1'],
        results_bert_7['eval_f1'],

        results_roberta_3['eval_f1'],
        results_roberta_5['eval_f1'],
        results_roberta_7['eval_f1'],

        results_deberta_3['eval_f1'],
        results_deberta_5['eval_f1'],
        results_deberta_7['eval_f1'],

        results_distilbert_3['eval_f1'],

        results_electra_3['eval_f1'],
        results_electra_5['eval_f1'],
        results_electra_7['eval_f1'],

        f1_score(y_test_80, y_pred_linear, average='weighted'),
        f1_score(y_test_80, y_pred_poly, average='weighted'),
        f1_score(y_test_80, y_pred_rbf, average='weighted'),
        f1_score(y_test_80, y_pred_vote, average='weighted')
    ]
}

df_results = pd.DataFrame(data)
print(df_results)

             Model  Accuracy  Precision    Recall  F1 Score
0         BERT (3)  0.821429   0.820791  0.821429  0.820938
1         BERT (5)  0.833929   0.833510  0.833929  0.832721
2         BERT (7)  0.835714   0.835266  0.835714  0.834593
3      RoBERTa (3)  0.808929   0.808081  0.808929  0.808184
4      RoBERTa (5)  0.833929   0.833653  0.833929  0.833766
5      RoBERTa (7)  0.819643   0.820993  0.819643  0.820104
6      DeBERTa (3)  0.787500   0.796565  0.787500  0.788964
7      DeBERTa (5)  0.787500   0.786588  0.787500  0.786836
8      DeBERTa (7)  0.812500   0.811713  0.812500  0.811136
9   DistilBERT (3)  0.850000   0.851390  0.850000  0.850421
10     ELECTRA (3)  0.835714   0.836298  0.835714  0.833802
11     ELECTRA (5)  0.826786   0.827753  0.826786  0.827139
12     ELECTRA (7)  0.585714   0.343061  0.585714  0.432690
13      SVM Linear  0.685714   0.681832  0.685714  0.676505
14        SVM Poly  0.614286   0.672363  0.614286  0.510999
15         SVM RBF  0.664286   0.665459 

In [98]:
df_results.round(3)

,Model,Accuracy,Precision,Recall,F1 Score
0,BERT (3),0.821,0.821,0.821,0.821
1,BERT (5),0.834,0.834,0.834,0.833
2,BERT (7),0.836,0.835,0.836,0.835
3,RoBERTa (3),0.809,0.808,0.809,0.808
4,RoBERTa (5),0.834,0.834,0.834,0.834
5,RoBERTa (7),0.820,0.821,0.820,0.820
6,DeBERTa (3),0.788,0.797,0.788,0.789
7,DeBERTa (5),0.788,0.787,0.788,0.787
8,DeBERTa (7),0.812,0.812,0.812,0.811
9,DistilBERT (3),0.850,0.851,0.850,0.850


BEST MODEL

In [99]:
best_model = df_results.loc[df_results['F1 Score'].idxmax()]
print("Best Model:\n", best_model)

Best Model:
 Model        DistilBERT (3)
Accuracy               0.85
Precision           0.85139
Recall                 0.85
F1 Score           0.850421
Name: 9, dtype: object
